In [158]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,Dataset
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer



In [159]:
df = pd.read_csv(r'C:\Users\Lenovo\OneDrive\Desktop\Data Structures and Algorithms\Deep Learning\Pytorch\2\twitter_training.csv')

In [160]:
df.head()

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [161]:
df.drop(columns = ['2401','Borderlands'],inplace=True)

In [162]:
df = df.rename(columns={
    'im getting on borderlands and i will murder you all ,': 'text',
    'Positive': 'sentiment'
})


In [163]:
df.head()

,sentiment,text
0,Positive,I am coming to the borders and I will kill you...
1,Positive,im getting on borderlands and i will kill you ...
2,Positive,im coming on borderlands and i will murder you...
3,Positive,im getting on borderlands 2 and i will murder ...
4,Positive,im getting into borderlands and i can murder y...


In [164]:
df.value_counts()

sentiment   text                                                                                                                                                                                                                                                              
Neutral     At the same time, despite the fact that there are currently some 100 million people living below the poverty line, most of them do not have access to health services and do not have access to health care, while most of them do not have access to health care.    57
            It is not the first time that the EU Commission has taken such a step.                                                                                                                                                                                                57
                                                                                                                                                                               

In [165]:
df['text'].value_counts()

text
At the same time, despite the fact that there are currently some 100 million people living below the poverty line, most of them do not have access to health services and do not have access to health care, while most of them do not have access to health care.    172
                                                                                                                                                                                                                                                                      172
It is not the first time that the EU Commission has taken such a step.                                                                                                                                                                                                172
<unk>                                                                                                                                                                                                

In [166]:
df['sentiment'].value_counts()

sentiment
Negative      22542
Positive      20831
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64

In [167]:
le = LabelEncoder()
df['sentiment']= le.fit_transform(df['sentiment'])

In [168]:
df

,sentiment,text
0,3,I am coming to the borders and I will kill you...
1,3,im getting on borderlands and i will kill you ...
2,3,im coming on borderlands and i will murder you...
3,3,im getting on borderlands 2 and i will murder ...
4,3,im getting into borderlands and i can murder y...
...,...,...
74676,3,Just realized that the Windows partition of my...
74677,3,Just realized that my Mac window partition is ...
74678,3,Just realized the windows partition of my Mac ...
74679,3,Just realized between the windows partition of...


In [169]:
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
texts = df['text'].fillna("").astype(str).tolist()

encodings = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)


In [170]:
encodings

{'input_ids': tensor([[  101,  1045,  2572,  ...,     0,     0,     0],
        [  101, 10047,  2893,  ...,     0,     0,     0],
        [  101, 10047,  2746,  ...,     0,     0,     0],
        ...,
        [  101,  2074,  3651,  ...,     0,     0,     0],
        [  101,  2074,  3651,  ...,     0,     0,     0],
        [  101,  2074,  2066,  ...,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])}

In [171]:
print(type(encodings))

<class 'transformers.tokenization_utils_base.BatchEncoding'>


In [172]:
encodings[0]

Encoding(num_tokens=128, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [173]:
input_ids = encodings["input_ids"]        # shape: [batch, 128]
attention_mask = encodings["attention_mask"]

In [174]:
print(type(input_ids))

<class 'torch.Tensor'>


In [175]:
input_ids.shape
attention_mask.shape

torch.Size([74681, 128])

In [176]:
labels = torch.tensor(df['sentiment'])

In [177]:
labels.shape

torch.Size([74681])

In [178]:
X = input_ids

In [179]:
print(type(X))

<class 'torch.Tensor'>


In [180]:
X.shape

torch.Size([74681, 128])

In [181]:
y= labels

In [182]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [183]:
X_train.shape

torch.Size([59744, 128])

In [184]:
class Twitte(Dataset):
    def __init__(self,input_ids,labels):
        self.input_ids = input_ids
        self.labels = labels
    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'label' : self.labels[idx]
        }

In [185]:
train_dataset = Twitte(X_train,y_train)
test_dataset = Twitte(X_test,y_test)

In [186]:
vocab_size = tokenizer.vocab_size
embed_dim = 128


In [187]:
input_ids=X_train.shape
output_dim = 4

In [188]:
print(type(input_ids))

<class 'torch.Size'>


In [189]:
class SimpleRNN(nn.Module):
    def __init__(self,hidden_layer,embed_dim, input_ids,output_dim,neuron_per_layer,dropout_rate):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embed_dim)
        self.rnn_layers = nn.ModuleList()
        for i in range(hidden_layer):
            input_ids = embed_dim if i == 0 else neuron_per_layer
            self.rnn_layers.append(
                nn.RNN(input_ids,neuron_per_layer,batch_first=True)
            )
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(neuron_per_layer,output_dim)
    def forward(self,X):
        X = self.embedding(X)
        for rnn in self.rnn_layers:
            X,_ = rnn(X)
            X = X[:,-1,:]
            X = self.dropout(X)
            out = self.fc(X)
            return X
            
        
    

In [190]:
device = torch.device('cuda' if torch.cuda.is_available else 'cpu')
print(device)

cuda


In [191]:
import torch.optim as optim

In [192]:
print(type(input_ids))

<class 'torch.Size'>


In [193]:
X_train.shape

torch.Size([59744, 128])

In [194]:
y_train.shape

torch.Size([59744])

In [206]:
def objective(trial):
    # Hyperparameter to tune
    hidden_layer = trial.suggest_int('hidden_layer',2,20,step=2)
    neuron_per_layer = trial.suggest_int('neuron_per_layer',10,200,step = 10)
    learning_rate = trial.suggest_float('learning_rate',1e-5,1e-1, log=True)
    dropout_rate = trial.suggest_float('dropout_rate',0.1,0.5,step = 0.1)
    epochs = trial.suggest_int('epochs',10,100,step = 10)
    optimizer_name = trial.suggest_categorical('optimizer_name',['SGD','Adam','RMSprop'])
    weight_decay = trial.suggest_float('wegiht_decay',1e-5,1e-1, log=True)
    batch_size= trial.suggest_int('batch_size',8,64,step = 4)

    
    # model inilization
    output_dim = 4
    model = SimpleRNN(hidden_layer=hidden_layer,embed_dim = 128, input_ids=vocab_size,output_dim = output_dim,neuron_per_layer = neuron_per_layer,dropout_rate = dropout_rate)
    model.to(device)
    
    #optimizer_selection
    optimizer= optim.SGD(model.parameters(),lr = learning_rate ,weight_decay = weight_decay)
    if optimizer_name == 'Adam':
        
        optimizer = optim.Adam(model.parameters(),lr = learning_rate,weight_decay = weight_decay)
    elif optimizer_name == 'SGD':
        optimizer = optim.SGD(model.parameters(),lr = learning_rate , weight_decay = weight_decay)
    else:
        
        optimizer = optim.RMSprop(model.parameters(),lr = learning_rate,weight_decay= weight_decay)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(train_dataset,batch_size = batch_size,shuffle=True)
    test_loader = DataLoader(test_dataset , batch_size = batch_size, shuffle=True)
    
    model.train()
    for epoch in range(epochs):
        for batch_data in train_loader:
            batch_texts = batch_data['input_ids']
            batch_labels = batch_data['label']
            
            # batch_texts may be a tensor or list - ensure it's a list of strings
            if isinstance(batch_texts, torch.Tensor):
                batch_texts = batch_texts.tolist()
            
            # Ensure all elements are strings
            batch_texts = [str(t) if not isinstance(t, str) else t for t in batch_texts]
            
            encodings = tokenizer(batch_texts, padding=True, truncation=True, return_tensors='pt')
            batch_input_ids = encodings['input_ids'].to(device)
            batch_labels = batch_labels.to(device).long()
            
            optimizer.zero_grad()
            outputs = model(batch_input_ids)
            loss = criterion(outputs, batch_labels)
            loss.backward()
            optimizer.step()
    # Accuracy
    model.eval()
    correct , total = 0,0
    with torch.no_grad():
        for batch_data in test_loader:
            batch_texts = batch_data['input_ids']
            batch_labels = batch_data['label']
            
            # batch_texts may be a tensor or list - ensure it's a list of strings
            if isinstance(batch_texts, torch.Tensor):
                batch_texts = batch_texts.tolist()
            
            # Ensure all elements are strings
            batch_texts = [str(t) if not isinstance(t, str) else t for t in batch_texts]
            
            encodings = tokenizer(batch_texts, padding=True, truncation=True, return_tensors='pt')
            batch_input_ids = encodings['input_ids'].to(device)
            batch_labels = batch_labels.to(device).long()
            
            outputs = model(batch_input_ids)
            preds = outputs.argmax(dim = 1)
            correct += (preds == batch_labels).sum().item()
            total+= batch_labels.size(0)
    accuracy = correct /total
    return accuracy

In [196]:
!pip install optuna

In [197]:
import optuna

study = optuna.create_study(direction='maximize')


[I 2026-01-11 21:30:42,362] A new study created in memory with name: no-name-1da0efc9-9915-4886-b57b-6b3a7469ad7f


In [198]:
print(type(input_ids))

<class 'torch.Size'>


In [207]:
study.optimize(objective, n_trials=20)

[I 2026-01-11 22:22:35,821] Trial 4 finished with value: 0.301399210015398 and parameters: {'hidden_layer': 10, 'neuron_per_layer': 20, 'learning_rate': 0.002075681749621946, 'dropout_rate': 0.5, 'epochs': 70, 'optimizer_name': 'SGD', 'wegiht_decay': 0.0033221689491309612, 'batch_size': 48}. Best is trial 4 with value: 0.301399210015398.
[I 2026-01-11 22:37:09,214] Trial 5 finished with value: 0.3030059583584388 and parameters: {'hidden_layer': 8, 'neuron_per_layer': 160, 'learning_rate': 0.0009092455197103, 'dropout_rate': 0.5, 'epochs': 10, 'optimizer_name': 'SGD', 'wegiht_decay': 0.0026921486586127244, 'batch_size': 8}. Best is trial 5 with value: 0.3030059583584388.
[I 2026-01-11 22:46:11,354] Trial 6 finished with value: 0.29932382673897034 and parameters: {'hidden_layer': 20, 'neuron_per_layer': 120, 'learning_rate': 0.0346652754221445, 'dropout_rate': 0.4, 'epochs': 10, 'optimizer_name': 'RMSprop', 'wegiht_decay': 9.681887207444374e-05, 'batch_size': 40}. Best is trial 5 with va

KeyboardInterrupt: 